Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [53]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

model=init_chat_model("google_genai:gemini-2.5-flash")
model


ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000164AFB573D0>, default_metadata=(), model_kwargs={})

### Summarization
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. 
Summarization is useful for the following:

-  Long-running conversations that exceed context windows.

-  Multi-turn dialogues with extensive history.

-  Applications where preserving full conversation context matters.

this is based on message length

In [34]:
from langchain_core import messages
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [35]:
### Run with thread id
config = {"configurable":{"thread_id":"test-1"}}


In [36]:

questions = [
    "What is docker?",
    "what is kubernetes?",
    "what is AWS?",
    "What is EC2?",
    "what is s3 bucket",
    "what is IAM ?"
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is docker?', additional_kwargs={}, response_metadata={}, id='beea5906-3b38-49eb-a745-d8c9b1dafd90'), AIMessage(content='**Docker** is a containerization platform that allows developers to package, ship, and run applications in containers. Containers are lightweight and portable, providing a consistent and reliable way to deploy applications across different environments.\n\n### Key Features of Docker\n\n* **Lightweight**: Containers are much lighter than traditional virtual machines, as they share the same kernel as the host operating system.\n* **Portable**: Containers are highly portable, allowing developers to deploy applications across different environments, including development, testing, staging, and production.\n* **Isolated**: Containers provide a high level of isolation between applications, ensuring that they do not interfere with each other.\n* **Efficient**: Containers are highly efficient, as they share resources and do n

this is based on token size

In [56]:
model.invoke("who are you?")

AIMessage(content='I am a large language model, trained by Google.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f9f3e-edd3-7350-a07a-3e40b1c0f6ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 508, 'total_tokens': 513, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 497}})

In [54]:
from langchain_core import messages
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",10000),
            keep = ("tokens",5000)
        )
    ]
)

In [55]:
### Run with thread id
config1 = {"configurable":{"thread_id":"test-2"}}

def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

questions = [
    "What is docker?",
    "what is kubernetes?",
    "what is AWS?",
    "What is EC2?",
    "what is s3 bucket",
    "what is IAM ?"
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config1)
    
    tokens = count_token(response["messages"])
    
    print(f"Messages: {len(response['messages'])}")
    print(f"Messages: {response}")
    print(tokens/4)
 

Messages: 2
Messages: {'messages': [HumanMessage(content='What is docker?', additional_kwargs={}, response_metadata={}, id='c93ac4a1-c4b4-4fc9-8189-7cf33e40a37f'), AIMessage(content='Imagine you\'re trying to pack a suitcase for a trip. You have your clothes, toiletries, shoes, and maybe some books. If you just throw everything in, it might get messy, things could break, and it might not fit well in a different suitcase.\n\n**Docker is like a standardized, super-efficient way to pack and ship your software.**\n\nHere\'s a breakdown:\n\n1.  **The Core Idea: Containers**\n    *   Instead of just your application code, a Docker container bundles your application **and everything it needs to run**: libraries, dependencies, configuration files, and even a miniature operating system environment.\n    *   Think of it like a **shipping container** for software. Just as a physical shipping container can hold anything (electronics, furniture, food) and be easily moved between ships, trains, and 